In [ ]:
import h5py
import numpy as np
import torch
from threading import Thread
from queue import Queue
import threading
import math
import psutil
import logging
from typing import Optional, Tuple

class RAMCacheLoader:
    def __init__(
        self,
        data_path: str,
        batch_size: int,
        world_size: int,
        target_ram_gb: float = 100,
        monitor_ram: bool = True
    ):
        """
        Initialize the RAM cache loader
        
        Args:
            data_path: Path to h5py file
            batch_size: Size of each batch
            world_size: Number of GPUs/processes
            target_ram_gb: Target RAM utilization in GB
            monitor_ram: Whether to log RAM usage stats
        """
        self.batch_size = batch_size
        self.world_size = world_size
        self.target_ram_bytes = target_ram_gb * (1024**3)
        self.monitor_ram = monitor_ram
        
        # Setup logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
        # Open dataset
        self.file = h5py.File(data_path, 'r')
        self.activations = self.file['activations']
        self.total_vectors = self.activations.shape[0]
        self.vector_size = self.activations.shape[1]
        
        # Calculate how many vectors we can cache in RAM
        bytes_per_value = 2  # bfloat16
        bytes_per_vector = self.vector_size * bytes_per_value
        self.max_cached_vectors = int(self.target_ram_bytes / bytes_per_vector)
        
        # Initialize cache state
        self.cache_start_idx = 0
        self.cached_data = None
        self.cache_lock = threading.Lock()
        
        # Setup background loading
        self.load_queue = Queue(maxsize=1)
        self.should_stop = False
        self.load_thread = Thread(target=self._background_loader, daemon=True)
        self.load_thread.start()
        
        # Load initial cache
        self._load_cache_chunk(0)
        
    def _get_ram_usage(self) -> Tuple[float, float]:
        """Get current RAM usage in GB and percentage"""
        ram = psutil.Process().memory_info().rss
        ram_gb = ram / (1024**3)
        ram_percent = psutil.virtual_memory().percent
        return ram_gb, ram_percent
        
    def _load_cache_chunk(self, start_idx: int) -> None:
        """Load a chunk of data into RAM cache"""
        end_idx = min(start_idx + self.max_cached_vectors, self.total_vectors)
        chunk = self.activations[start_idx:end_idx]
        
        with self.cache_lock:
            self.cached_data = torch.from_numpy(chunk).view(dtype=torch.bfloat16)
            self.cache_start_idx = start_idx
            
        if self.monitor_ram:
            ram_gb, ram_percent = self._get_ram_usage()
            self.logger.info(
                f"Loaded cache chunk {start_idx:,}-{end_idx:,}. "
                f"RAM usage: {ram_gb:.1f}GB ({ram_percent:.1f}%)"
            )
    
    def _background_loader(self) -> None:
        """Background thread that manages cache loading"""
        while not self.should_stop:
            try:
                next_start_idx = self.load_queue.get()
                if next_start_idx is None:  # Shutdown signal
                    break
                self._load_cache_chunk(next_start_idx)
            except Exception as e:
                self.logger.error(f"Error in background loader: {e}")
                
    def _request_next_chunk(self) -> None:
        """Request loading of next chunk in background"""
        next_start = (self.cache_start_idx + self.max_cached_vectors) % self.total_vectors
        self.load_queue.put(next_start)
        
    def get_batches(self) -> torch.Tensor:
        """Get world_size batches from cache, trigger background load if needed"""
        total_samples = self.batch_size * self.world_size
        
        with self.cache_lock:
            # Calculate indices within cache
            cache_offset = (self.current_idx - self.cache_start_idx) % self.max_cached_vectors
            cache_end = min(cache_offset + total_samples, len(self.cached_data))
            
            # If we need to load next chunk
            if cache_end - cache_offset < total_samples:
                self._request_next_chunk()
                
            # Get batches from cache
            batches = self.cached_data[cache_offset:cache_offset + total_samples]
            
        self.current_idx += total_samples
        return batches.view(self.world_size, self.batch_size, -1)
        
    def __iter__(self):
        """Return iterator over batches"""
        self.current_idx = 0
        return self
        
    def __next__(self) -> torch.Tensor:
        """Get next set of batches"""
        if self.current_idx >= self.total_vectors:
            raise StopIteration
            
        return self.get_batches()
        
    def __len__(self) -> int:
        """Return total number of batch sets"""
        return math.ceil(self.total_vectors / (self.batch_size * self.world_size))
        
    def close(self) -> None:
        """Cleanup resources"""
        self.should_stop = True
        self.load_queue.put(None)  # Signal background thread to stop
        self.load_thread.join()
        self.file.close()

# Example usage:
if __name__ == "__main__":
    loader = RAMCacheLoader(
        data_path='/home/henry/Documents/PythonProjects/open-concept-steering/data/residual_stream_activations_llama1b_bf16.h5',
        batch_size=4096,
        world_size=3,
        target_ram_gb=100
    )
    
    try:
        for batches in loader:
            # batches will be shape (world_size, batch_size, vector_size)
            # Process batches with your GPUs
            print(batches)
            batches.to('cuda')
    finally:
        loader.close()

In [ ]:
import h5py
import time
f = h5py.File("/home/henry/Documents/PythonProjects/open-concept-steering/data/residual_stream_activations_llama1b_bf16.h5",'r')
activations = f['activations']
num_vectors = len(activations)

#time how long it takes to load a quarter of the dataset
start = time.time()
vectors = activations[:num_vectors//4]
print(f"without memmap took {time.time() - start} to load {num_vectors//4} vectors")


TypeError: unsupported operand type(s) for -: 'builtin_function_or_method' and 'float'

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, DataLoader
import h5py
from typing import Optional, Tuple

class SAE(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.encode = nn.Linear(input_size, hidden_size)
        self.decode = nn.Linear(hidden_size, input_size)
        
    def forward(self, x):
        features = self.encode(x)
        features = F.relu(features)
        reconstruction = self.decode(features)
        return reconstruction, features
    
    def get_decoder_norms(self):
        return torch.linalg.vector_norm(self.decode.weight, dim=0)


class ResidualStreamDataset(Dataset):
    def __init__(self, path: str, num_examples: Optional[int] = None):
        self.file = h5py.File(path, 'r')
        self.activations = self.file['activations']
        if num_examples is not None:
            self.activations = self.activations[:num_examples]
        
    def __len__(self):
        return len(self.activations)
    
    def __getitem__(self, idx):
        return torch.from_numpy(self.activations[idx]).view(torch.bfloat16)
    
    def __del__(self):
        self.file.close()

def get_dataloader(
    path: str,
    batch_size: int,
    num_examples: Optional[int] = None,
    num_workers: int = 4,
    shuffle: bool = True
) -> DataLoader:
    dataset = ResidualStreamDataset(path, num_examples)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True
    )

def train_sae(
    data_path: str,
    input_size: int = 2048,
    hidden_size: int = 8192,
    batch_size: int = 32,
    num_epochs: int = 1000,
    lr: float = 1e-3,
    lambda_max: float = 5.0,
    warmup_fraction: float = 0.2,
    num_workers: int = 4,
    num_batches_to_overfit: int = 10  # New parameter
):
    # Setup data
    train_loader = get_dataloader(
        data_path,
        batch_size=batch_size,
        num_workers=num_workers
    )
    
    # Get the batches we want to overfit
    overfit_batches = []
    for i, batch in enumerate(train_loader):
        if i >= num_batches_to_overfit:
            break
        overfit_batches.append(batch.cuda())
    
    # Initialize model
    model = SAE(input_size, hidden_size).cuda().to(torch.bfloat16)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training loop
    lambda_l1 = 0  # Start at 0 and increase during warmup
    warmup_epochs = int(num_epochs * warmup_fraction)
    
    from tqdm import tqdm
    for epoch in tqdm(range(num_epochs)):
        total_loss = 0
        total_mse = 0
        total_l1 = 0
        
        # Train on our few fixed batches
        for batch in overfit_batches:
            reconstruction, features = model(batch)
            
            # Losses
            mse_loss = F.mse_loss(batch, reconstruction)
            decoder_norms = model.get_decoder_norms()
            l1_loss = torch.sum(torch.abs(features) * decoder_norms[None, :]) / (batch.shape[0] * features.shape[1])
            loss = mse_loss + (lambda_l1 * l1_loss)
            
            # Optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track metrics
            total_loss += loss.item()
            total_mse += mse_loss.item()
            total_l1 += l1_loss.item()
        
        # Update lambda during warmup
        if epoch < warmup_epochs:
            lambda_l1 = (epoch / warmup_epochs) * lambda_max
            
        # Log metrics
        if epoch % 100 == 0:
            avg_loss = total_loss / len(overfit_batches)
            avg_mse = total_mse / len(overfit_batches)
            avg_l1 = total_l1 / len(overfit_batches)
            
            with torch.no_grad():
                # Get metrics from last batch
                mean_activation = features.abs().mean().item()
                sparsity = (features > 0).float().mean().item()
                active_features = torch.linalg.vector_norm(features, ord=0, dim=0).sum().item()/batch_size
                recon_error = torch.linalg.vector_norm(reconstruction - batch) / torch.linalg.vector_norm(batch)
            
            print(f"Epoch {epoch}, Total Loss: {avg_loss:.6f}")
            print(f"- MSE Loss: {avg_mse:.6f}")
            print(f"- L1 Loss: {avg_l1:.6f}")
            print(f"- Mean activation: {mean_activation:.4f}")
            print(f"- Sparsity: {sparsity:.4f}")
            print(f"- Active features: {active_features:.1f}")
            print(f"- Reconstruction error: {recon_error:.4f}")
            print("---")
    
    return model

if __name__ == "__main__":
    # Training settings
    settings = {
        "data_path": '/home/henry/Documents/PythonProjects/open-concept-steering-dataset/residual_stream_activations_llama1b_bf16.h5',
        "input_size": 2048,
        "hidden_size": 8192*4,
        "batch_size": 32,
        "num_epochs": 1000,
        "lr": 1e-3,
        "lambda_max": 5.0,
        "warmup_fraction": 0.2,
        "num_workers": 24
    }
    
    model = train_sae(**settings)

FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = '/home/henry/Documents/PythonProjects/open-concept-steering-dataset/residual_stream_activations_llama1b_bf16.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, DataLoader
import h5py
from typing import Optional, Tuple

class SAE(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.encode = nn.Linear(input_size, hidden_size)
        self.decode = nn.Linear(hidden_size, input_size)
        
    def forward(self, x):
        features = self.encode(x)
        features = F.relu(features)
        reconstruction = self.decode(features)
        return reconstruction, features
    
    def get_decoder_norms(self):
        return torch.linalg.vector_norm(self.decode.weight, dim=0)


class ResidualStreamDataset(Dataset):
    def __init__(self, path: str, num_examples: Optional[int] = None):
        self.file = h5py.File(path, 'r')
        self.activations = self.file['activations']
        if num_examples is not None:
            self.activations = self.activations[:num_examples]
        
    def __len__(self):
        return len(self.activations)
    
    def __getitem__(self, idx):
        return torch.from_numpy(self.activations[idx]).view(torch.bfloat16).to(torch.float32) #load from uint16 to bfloat16, then cast to float32. there must be a better way
    
    def __del__(self):
        self.file.close()

def get_dataloader(
    path: str,
    batch_size: int,
    num_examples: Optional[int] = None,
    num_workers: int = 4,
    shuffle: bool = True
) -> DataLoader:
    dataset = ResidualStreamDataset(path, num_examples)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True
    )

def train_sae(
    data_path: str,
    input_size: int = 2048,
    hidden_size: int = 8192,
    batch_size: int = 32,
    num_epochs: int = 1000,
    lr: float = 1e-3,
    lambda_max: float = 5.0,
    warmup_fraction: float = 0.2,
    num_workers: int = 4,
    num_batches_to_overfit: int = 10
):
    # Setup data
    train_loader = get_dataloader(
        data_path,
        batch_size=batch_size,
        num_workers=num_workers
    )
    
    # Get the batches we want to overfit
    overfit_batches = []
    for i, batch in enumerate(train_loader):
        if i >= num_batches_to_overfit:
            break
        overfit_batches.append(batch.cuda())
    
    # Initialize model
    model = SAE(input_size, hidden_size).cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training loop
    lambda_l1 = 0
    warmup_epochs = int(num_epochs * warmup_fraction)
    
    from tqdm import tqdm
    for epoch in tqdm(range(num_epochs)):
        total_loss = 0
        total_mse = 0
        total_l1 = 0
        
        # Train on our fixed batches
        for batch in overfit_batches:
            # Use autocast for mixed precision
            with torch.autocast("cuda", dtype=torch.bfloat16):
                reconstruction, features = model(batch)
                
                # Losses
                mse_loss = F.mse_loss(batch, reconstruction)
                decoder_norms = model.get_decoder_norms()
                l1_loss = torch.sum(torch.abs(features) * decoder_norms[None, :]) / (batch.shape[0] * features.shape[1])
                loss = mse_loss + (lambda_l1 * l1_loss)
            
            # Optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track metrics
            total_loss += loss.item()
            total_mse += mse_loss.item()
            total_l1 += l1_loss.item()
        
        # Update lambda during warmup
        if epoch < warmup_epochs:
            lambda_l1 = (epoch / warmup_epochs) * lambda_max
            
        # Log metrics
        if epoch % 100 == 0:
            avg_loss = total_loss / len(overfit_batches)
            avg_mse = total_mse / len(overfit_batches)
            avg_l1 = total_l1 / len(overfit_batches)
            
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                # Get metrics from last batch
                mean_activation = features.abs().mean().item()
                sparsity = (features > 0).float().mean().item()
                active_features = torch.linalg.vector_norm(features, ord=0, dim=0).sum().item()/batch_size
                recon_error = torch.linalg.vector_norm(reconstruction - batch) / torch.linalg.vector_norm(batch)
            
            print(f"Epoch {epoch}, Total Loss: {avg_loss:.6f}")
            print(f"- MSE Loss: {avg_mse:.6f}")
            print(f"- L1 Loss: {avg_l1:.6f}")
            print(f"- Mean activation: {mean_activation:.4f}")
            print(f"- Sparsity: {sparsity:.4f}")
            print(f"- Active features: {active_features:.1f}")
            print(f"- Reconstruction error: {recon_error:.4f}")
            print("---")
    
    return model

if __name__ == "__main__":
    # Training settings
    settings = {
        "data_path": '/home/henry/Documents/PythonProjects/open-concept-steering-dataset/residual_stream_activations_llama1b_bf16.h5',
        "input_size": 2048,
        "hidden_size": 8192*4,
        "batch_size": 32,
        "num_epochs": 1000,
        "lr": 1e-3,
        "lambda_max": 5.0,
        "warmup_fraction": 0.2,
        "num_workers": 48
    }
    
    model = train_sae(**settings)

  0%|          | 2/1000 [00:00<06:13,  2.67it/s]

Epoch 0, Total Loss: 51.287574
- MSE Loss: 51.287574
- L1 Loss: 0.007303
- Mean activation: 0.0312
- Sparsity: 0.3893
- Active features: 12756.1
- Reconstruction error: 18.1872
---


 10%|█         | 102/1000 [00:18<02:41,  5.55it/s]

Epoch 100, Total Loss: 0.011492
- MSE Loss: 0.008546
- L1 Loss: 0.001190
- Mean activation: 0.0025
- Sparsity: 0.0486
- Active features: 1593.9
- Reconstruction error: 0.1252
---


 20%|██        | 202/1000 [00:36<02:23,  5.55it/s]

Epoch 200, Total Loss: 0.002453
- MSE Loss: 0.000128
- L1 Loss: 0.000467
- Mean activation: 0.0003
- Sparsity: 0.0047
- Active features: 154.9
- Reconstruction error: 0.0423
---


 22%|██▏       | 222/1000 [00:40<02:20,  5.54it/s]